<a href="https://colab.research.google.com/github/roasfora/open_library_facultet_example/blob/test/Instructor_Books_API_Git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INSTRUCTOR | Mini-project: Books API + pandas + GitHub

**Lesson:** Data Processing Pipeline + Git Workflow (16 November 2026)  
**Level:** Introductory Data Science  
**Goal:** Search for books using the free Open Library API, make a clean DataFrame, save a CSV, and publish your notebook on GitHub.

**Workflow:** API (JSON) → pandas → clean → CSV → Git commit → Git push.

Work individually or in pairs. **No API key is required.** Make only one API request when running the collection cell; avoid repeatedly running the whole notebook.


## Instructor notes / pacing (55–65 minutes)

- **0–8 min:** Review workflow vs pipeline, GET request, JSON and `docs`.
- **8–25 min:** Run API, inspect JSON, build and clean DataFrame.
- **25–35 min:** Export and verify CSV; students modify query and rerun.
- **35–50 min:** Demonstrate `status → add → commit → push`; partner pulls/reviews.
- **50–65 min (optional):** Use branches and pull requests in pairs.

**Reliability:** Open Library is keyless but has fair-use limits. This notebook makes one request per run, uses `fields` and `limit`, and includes an explicitly labeled offline sample. Ask students not to repeatedly refresh the request cell.

**Git safety:** Do not run `git init` inside an already-cloned repository. Before committing, inspect `git status` and stage only your intended files. Avoid checking tokens/passwords into Git. No API key is needed here.

**Teaching distinction:** A workflow is the sequence of steps; a reusable pipeline arranges those steps so they run consistently. A notebook with sequential cells can illustrate both; wrapping steps in functions is an extension, not necessary for beginners.

## 0. Install and import libraries

If needed, run `%pip install requests pandas`. Run the notebook in VS Code or Google Colab. If offline, place `sample_books.json` next to this notebook; a sample data fallback is included. The sample is **illustrative**, not live API data.

In [12]:
# Uncomment only if your environment needs them:
# %pip install requests pandas

import json
from pathlib import Path
import requests
import pandas as pd
from IPython.display import display

## 1. Collect data from an API

`requests.get()` sends a GET request. `params` adds the query string safely; `.json()` converts JSON to Python objects. `raise_for_status()` helps detect HTTP errors. The API response has a key named **`docs`** containing a list of book records.

**Try:** Change `SEARCH_TERM` to `machine learning` or `data science`.

In [13]:
URL = "https://openlibrary.org/search.json"
SEARCH_TERM = "java programming"  # Try another topic!
PARAMS = {
    "q": SEARCH_TERM,
    "fields": "title,author_name,first_publish_year,edition_count",
    "limit": 12,
}

try:
    response = requests.get(
        URL, params=PARAMS, timeout=20,
        headers={"User-Agent": "DS4IntroClass/1.0 (educational notebook)"},
    )
    response.raise_for_status()
    data = response.json()
    source = "LIVE Open Library API"
except requests.RequestException as exc:
    print("API unavailable:", exc)
    print("Loading the illustrative sample_books.json file instead.")
    with open("sample_books.json", encoding="utf-8") as file:
        data = json.load(file)
    source = "OFFLINE SAMPLE (not live search results)"

print("Source:", source)
print("Number of records received:", len(data["docs"]))
print("First record:", data["docs"][0] if data["docs"] else "No records")

Source: LIVE Open Library API
Number of records received: 12
First record: {'author_name': ['Y. Daniel Liang'], 'edition_count': 12, 'first_publish_year': 1999, 'title': 'Introduction to Java Programming'}


In [14]:
print(data)

{'numFound': 5197, 'start': 0, 'numFoundExact': True, 'num_found': 5197, 'documentation_url': 'https://openlibrary.org/dev/docs/api/search', 'q': 'java programming', 'offset': None, 'docs': [{'author_name': ['Y. Daniel Liang'], 'edition_count': 12, 'first_publish_year': 1999, 'title': 'Introduction to Java Programming'}, {'author_name': ['Joyce Farrell'], 'edition_count': 4, 'first_publish_year': 1998, 'title': 'Java Programming'}, {'author_name': ['Joshua Bloch'], 'edition_count': 10, 'first_publish_year': 2001, 'title': 'Effective Java'}, {'author_name': ['Cay S. Horstmann'], 'edition_count': 26, 'first_publish_year': 2001, 'title': 'Big Java'}, {'author_name': ['Arnold, Ken'], 'edition_count': 5, 'first_publish_year': 1996, 'title': 'The Java programming language'}, {'author_name': ['D. S. Malik'], 'edition_count': 6, 'first_publish_year': 2004, 'title': 'Java programming'}, {'author_name': ['Joyce Farrell'], 'edition_count': 3, 'first_publish_year': 2009, 'title': 'Java programming

## 2. Convert JSON to a pandas DataFrame

The response is a dictionary. The book records are in `data["docs"]` (a **list of dictionaries**).

In [15]:
df = pd.DataFrame(data["docs"])
display(df.head())

,author_name,edition_count,first_publish_year,title
0,[Y. Daniel Liang],12,1999,Introduction to Java Programming
1,[Joyce Farrell],4,1998,Java Programming
2,[Joshua Bloch],10,2001,Effective Java
3,[Cay S. Horstmann],26,2001,Big Java
4,"[Arnold, Ken]",5,1996,The Java programming language


## 3. Transform and clean the data

We keep only four columns, convert the author list to readable text, fill missing publication years, and sort the rows. This is the **processing** part of the pipeline.

In [16]:
# Make sure these columns exist, even if a record is missing a field.
for column in ["title", "author_name", "first_publish_year", "edition_count"]:
    if column not in df.columns:
        df[column] = None

books = df[["title", "author_name", "first_publish_year", "edition_count"]].copy()
books["author"] = books["author_name"].apply(
    lambda names: ", ".join(names) if isinstance(names, list) else "Unknown"
)
books["year"] = pd.to_numeric(books["first_publish_year"], errors="coerce")
books["editions"] = pd.to_numeric(books["edition_count"], errors="coerce")
books = books[["title", "author", "year", "editions"]]
books = books.dropna(subset=["title"])
books = books.sort_values("title").reset_index(drop=True)

display(books.head(10))

,title,author,year,editions
0,Big Java,Cay S. Horstmann,2001,26
1,Building Java programs,Stuart Reges,2011,11
2,Effective Java,Joshua Bloch,2001,10
3,Introduction to Java Programming,Y. Daniel Liang,1999,12
4,Introduction to Java Programming and Data Stru...,"Y. Liang, Y. Daniel Liang",2019,1
5,"Introduction to Java Programming, Comprehensive",Y. Daniel Liang,2004,3
6,Java Programming,Joyce Farrell,1998,4
7,Java Programming,K. N. King,2000,2
8,Java Programming,Gilbert Grunitzky,2017,1
9,Java programming,D. S. Malik,2004,6


## 4. Save a CSV

A CSV is an easy-to-share table. `index=False` avoids adding an unwanted index column. This file is created in your current notebook working directory.

In [17]:
OUTPUT_FILE = "books_clean.csv"
books.to_csv(OUTPUT_FILE, index=False)
print("Saved:", OUTPUT_FILE)

# Verify that the file was really saved:
display(pd.read_csv(OUTPUT_FILE).head())

Saved: books_clean.csv


,title,author,year,editions
0,Big Java,Cay S. Horstmann,2001,26
1,Building Java programs,Stuart Reges,2011,11
2,Effective Java,Joshua Bloch,2001,10
3,Introduction to Java Programming,Y. Daniel Liang,1999,12
4,Introduction to Java Programming and Data Stru...,"Y. Liang, Y. Daniel Liang",2019,1


### Optional: Download the CSV in Google Colab

Run only if using Colab. In VS Code, locate the CSV in your project folder.

In [18]:
# Only run in Google Colab:
# from google.colab import files
# files.download("books_clean.csv")

## 5. Share with Git and GitHub (VS Code terminal)

**First time only:** Create a repository on GitHub, then clone it locally with `git clone <YOUR-REPO-URL>`. Put this notebook and `sample_books.json` inside the repository folder before opening it in VS Code.

Work in an existing repository folder. **Avoid editing the same notebook simultaneously**; each student can use their own branch.

```bash
git status
git pull origin main
git switch -c books-yourname   # optional: create your own branch
git add Student_Books_API_Git.ipynb books_clean.csv
git commit -m "Add cleaned book data workflow"
git push -u origin books-yourname  # if on your branch
```

If you are working directly on `main` in your own repository, replace the last line with `git push origin main`. In a shared project, open a **Pull Request** on GitHub and let your partner review it. If you cannot push, ask for collaborator access or **fork** the repository.

**Important:** `commit` is local. `push` uploads commits to GitHub. `pull` receives teammates' updates. A pull request is a review request, not a Git command.

## 6. Student challenges

1. Change `SEARCH_TERM` and rerun the notebook. What changed in your DataFrame?
2. Add a filter to keep books published in **2010 or later** (use `books[books["year"] >= 2010]`). Display the result.
3. Save the filtered table as `books_recent.csv`.
4. In the terminal, run `git status`. Commit and push your notebook and CSV.
5. **Pair challenge:** Student A changes the API topic on their own branch; Student B reviews the change in a GitHub Pull Request. Then switch roles.

**Deliverables:** notebook (`.ipynb`), CSV (`.csv`), repository URL, and a screenshot or link to one commit or pull request.


In [19]:
# Instructor solution to the optional challenge
recent = books[books["year"] >= 2010].copy()
display(recent)
recent.to_csv("books_recent.csv", index=False)
print("Recent books saved:", len(recent))

,title,author,year,editions
1,Building Java programs,Stuart Reges,2011,11
4,Introduction to Java Programming and Data Stru...,"Y. Liang, Y. Daniel Liang",2019,1
8,Java Programming,Gilbert Grunitzky,2017,1


Recent books saved: 3


## Optional instructor extension: three tiny reusable functions

Demonstrate only if the class moves quickly. The **student notebook does not require functions**.

In [20]:
def collect_books(topic):
    response = requests.get(
        URL,
        params={"q": topic, "fields": PARAMS["fields"], "limit": 12},
        timeout=20,
        headers={"User-Agent": "DS4IntroClass/1.0 (educational notebook)"},
    )
    response.raise_for_status()
    return response.json()["docs"]


def transform_books(records):
    raw = pd.DataFrame(records)
    for col in ["title", "author_name", "first_publish_year", "edition_count"]:
        if col not in raw:
            raw[col] = None
    return pd.DataFrame({
        "title": raw["title"],
        "author": raw["author_name"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "Unknown"),
        "year": pd.to_numeric(raw["first_publish_year"], errors="coerce"),
        "editions": pd.to_numeric(raw["edition_count"], errors="coerce"),
    }).dropna(subset=["title"]).sort_values("title").reset_index(drop=True)


def save_books(table, filename="books_pipeline.csv"):
    table.to_csv(filename, index=False)
    return filename

# Optional use (makes a new API request; don't run repeatedly):
# records = collect_books("data science")
# filename = save_books(transform_books(records))
# print(filename)